# California Housing Prediction

**Tabular Regression Project** — Predict California median house values.

Models: CatBoost · LightGBM · XGBoost · FLAML · LazyPredict
Dataset: California Housing (17,000 rows × 9 columns)
Target: `median_house_value`

## 2 · Project Overview

We predict **median house values** in California districts using geographic, demographic, and housing stock features. With 17,000 rows, this is large enough for gradient-boosting models to excel while still being fast to train.

Unlike the Boston dataset, there's no target censoring issue here, and the geographic features (latitude/longitude) add spatial structure.

## 3 · Learning Objectives

1. Work with geographic features (lat/lon) in tabular ML.
2. Handle missing values (total_bedrooms has nulls).
3. Engineer per-household and per-room features.
4. Compare models on a mid-size dataset.
5. Use LazyPredict and FLAML for benchmarking.

## 4 · Problem Statement

Predict the **median house value** for California districts using census features including location, housing age, rooms, bedrooms, population, households, and income.

## 5 · Why This Project Matters

- **Housing affordability** is a critical policy issue in California.
- Geographic features teach how location affects prices.
- Per-household derived features demonstrate practical feature engineering.
- Large enough dataset for realistic model comparison.

## 6 · Dataset Overview

| Property | Value |
|----------|-------|
| **Rows** | 17,000 |
| **Columns** | 9 |
| **Features** | longitude, latitude, housing_median_age, total_rooms, total_bedrooms, population, households, median_income |
| **Target** | median_house_value |
| **Missing** | total_bedrooms has some nulls |
| **File** | `california_housing_train.csv` (local) |

## 7 · Dataset Source and License Notes

- **Source**: 1990 California Census data, popularized by Pace & Barry (1997).
- **License**: Public domain.
- **Note**: Values are capped at $500,001.

## 8 · Environment Setup

In [ ]:
import subprocess, sys

def _install_if_missing(pkg, import_name=None):
    try:
        __import__(import_name or pkg)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

for pkg, imp in [('catboost', None), ('lightgbm', None), ('xgboost', None),
                 ('flaml', None), ('lazypredict', None)]:
    _install_if_missing(pkg, imp)

print('All packages ready.')

## 9 · Imports

In [ ]:
import os, warnings, json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score
try:
    from sklearn.metrics import root_mean_squared_error
except ImportError:
    from sklearn.metrics import mean_squared_error
    def root_mean_squared_error(y_true, y_pred): return mean_squared_error(y_true, y_pred, squared=False)

warnings.filterwarnings('ignore')
SAVE_DIR = os.path.dirname(os.path.abspath('__file__'))
print('Imports complete.')

## 10 · Configuration / Constants

In [ ]:
SEED = 42
TEST_SIZE = 0.2
TARGET = 'median_house_value'
np.random.seed(SEED)

## 11 · Dataset Download or Loading

In [ ]:
DATA_PATH = os.path.join(SAVE_DIR, 'california_housing_train.csv')
assert os.path.exists(DATA_PATH), f'Dataset not found at {DATA_PATH}'
df = pd.read_csv(DATA_PATH)
print(f'Loaded: {df.shape}')
df.head()

## 12 · Data Validation Checks

In [ ]:
print('Missing values:')
print(df.isnull().sum())
print(f'\nDuplicates: {df.duplicated().sum()}')
print(f'Target range: [{df[TARGET].min()}, {df[TARGET].max()}]')
print(f'Target mean: {df[TARGET].mean():.0f}')

## 13 · Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0,0].scatter(df['median_income'], df[TARGET], alpha=0.2, s=3)
axes[0,0].set_xlabel('Median Income'); axes[0,0].set_ylabel(TARGET)
axes[0,0].set_title('Income vs House Value')

axes[0,1].scatter(df['longitude'], df['latitude'], c=df[TARGET], cmap='viridis', alpha=0.3, s=3)
axes[0,1].set_xlabel('Longitude'); axes[0,1].set_ylabel('Latitude')
axes[0,1].set_title('Geographic Price Distribution')

df[TARGET].hist(bins=50, ax=axes[1,0], edgecolor='black')
axes[1,0].set_title('House Value Distribution')

df['housing_median_age'].hist(bins=30, ax=axes[1,1], edgecolor='black')
axes[1,1].set_title('Housing Age Distribution')

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'eda_plots.png'), dpi=100)
plt.show()

## 14 · Target Analysis

In [ ]:
print(df[TARGET].describe())
print(f'\nSkewness: {df[TARGET].skew():.2f}')
print(f'Capped at 500001: {(df[TARGET] >= 500001).sum()} rows')

## 15 · Train / Validation / Test Split

In [ ]:
# Handle missing values first
df['total_bedrooms'] = df['total_bedrooms'].fillna(df['total_bedrooms'].median())

X = df.drop(columns=[TARGET])
y = df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=TEST_SIZE, random_state=SEED)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

## 16 · Preprocessing Strategy

All features are numeric. We fill missing `total_bedrooms` with the median. For tree models, no scaling needed.

## 17 · Feature Engineering

We derive per-household features that capture density and space.

In [ ]:
for df_part in [X_train, X_test]:
    df_part['rooms_per_household'] = df_part['total_rooms'] / (df_part['households'] + 1)
    df_part['bedrooms_per_room'] = df_part['total_bedrooms'] / (df_part['total_rooms'] + 1)
    df_part['population_per_household'] = df_part['population'] / (df_part['households'] + 1)
    df_part['income_per_age'] = df_part['median_income'] / (df_part['housing_median_age'] + 1)

print(f'Features after engineering: {X_train.shape[1]}')

## 18 · Baseline Model

In [ ]:
baseline = LinearRegression()
baseline.fit(X_train, y_train)
y_pred_base = baseline.predict(X_test)

baseline_rmse = root_mean_squared_error(y_test, y_pred_base)
baseline_mae = mean_absolute_error(y_test, y_pred_base)
baseline_r2 = r2_score(y_test, y_pred_base)
print(f'Baseline LR: RMSE={baseline_rmse:.0f}, MAE={baseline_mae:.0f}, R²={baseline_r2:.4f}')

## 19 · LazyPredict Benchmark

In [ ]:
from lazypredict.Supervised import LazyRegressor

# Sample for speed
X_train_lp = X_train.head(5000)
y_train_lp = y_train.head(5000)
X_test_lp = X_test.head(1000)
y_test_lp = y_test.head(1000)

lazy = LazyRegressor(verbose=0, ignore_warnings=True, random_state=SEED)
lazy_models, _ = lazy.fit(X_train_lp, X_test_lp, y_train_lp, y_test_lp)
print(lazy_models.head(15))

## 20 · FLAML AutoML Run

In [ ]:
try:
    from flaml import AutoML
    flaml_model = AutoML()
    flaml_model.fit(
        X_train, y_train,
        task='regression',
        time_budget=60,
        metric='rmse',
        seed=SEED,
        verbose=0
    )
    y_pred_flaml = flaml_model.predict(X_test)
    flaml_rmse = root_mean_squared_error(y_test, y_pred_flaml)
    flaml_mae = mean_absolute_error(y_test, y_pred_flaml)
    flaml_r2 = r2_score(y_test, y_pred_flaml)
    print(f'FLAML best: {flaml_model.best_estimator}')
    print(f'  RMSE: {flaml_rmse:.2f}')
    print(f'  MAE:  {flaml_mae:.2f}')
    print(f'  R²:   {flaml_r2:.4f}')
except Exception as e:
    print(f'FLAML failed: {e}')
    flaml_rmse = flaml_mae = flaml_r2 = None

## 21 · Additional Best-Library Workflow (CatBoost + LightGBM + XGBoost)

In [ ]:
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor

models = {
    'CatBoost': CatBoostRegressor(iterations=500, learning_rate=0.1, depth=6,
                                   random_seed=SEED, verbose=0),
    'LightGBM': LGBMRegressor(n_estimators=500, learning_rate=0.1, max_depth=6,
                              random_state=SEED, verbose=-1),
    'XGBoost': XGBRegressor(n_estimators=500, learning_rate=0.1, max_depth=6,
                            random_state=SEED, verbosity=0)
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    results[name] = {
        'RMSE': root_mean_squared_error(y_test, preds),
        'MAE': mean_absolute_error(y_test, preds),
        'R2': r2_score(y_test, preds)
    }
    print(f'{name}: RMSE={results[name]["RMSE"]:.2f}, MAE={results[name]["MAE"]:.2f}, R²={results[name]["R2"]:.4f}')

## 22 · Top 3 Model Selection

In [ ]:
all_results = {}
all_results['Baseline_LR'] = {'RMSE': baseline_rmse, 'MAE': baseline_mae, 'R2': baseline_r2}
if flaml_r2 is not None:
    all_results['FLAML'] = {'RMSE': flaml_rmse, 'MAE': flaml_mae, 'R2': flaml_r2}
all_results.update(results)

ranked = sorted(all_results.items(), key=lambda x: x[1]['RMSE'])
print('All models ranked by RMSE:')
for name, m in ranked:
    print(f"  {name:20s}  RMSE={m['RMSE']:.2f}  MAE={m['MAE']:.2f}  R²={m['R2']:.4f}")

top3_names = [r[0] for r in ranked[:3]]
print(f'\nTop 3: {top3_names}')

## 23 · Final Training and Evaluation of Top 3

In [ ]:
print('Top 3 Final Results:')
print('=' * 60)
for name in top3_names:
    m = all_results[name]
    print(f"{name}:")
    print(f"  RMSE: {m['RMSE']:.2f}")
    print(f"  MAE:  {m['MAE']:.2f}")
    print(f"  R²:   {m['R2']:.4f}")
    print()

## 24 · Error Analysis

In [ ]:
best_name = top3_names[0]
if best_name in models:
    best_model = models[best_name]
else:
    best_model = models['CatBoost']

y_pred_best = best_model.predict(X_test)
residuals = y_test.values - y_pred_best

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].scatter(y_pred_best, residuals, alpha=0.5)
axes[0].axhline(0, color='r', linestyle='--')
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('Residual')
axes[0].set_title('Residuals vs Predicted')

axes[1].hist(residuals, bins=30, edgecolor='black')
axes[1].set_title('Residual Distribution')

axes[2].scatter(y_test, y_pred_best, alpha=0.5)
axes[2].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
axes[2].set_xlabel('Actual'); axes[2].set_ylabel('Predicted')
axes[2].set_title('Actual vs Predicted')

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'error_analysis.png'), dpi=100)
plt.show()

## 25 · Interpretation and Business Insight

- **Median income** is by far the strongest predictor of house value.
- **Location** (lat/lon) captures neighborhood effects — coastal areas command premiums.
- Per-household features (rooms_per_household, bedrooms_per_room) add predictive signal.
- California housing prices are driven by income, location, and housing density.

## 26 · Limitations

- 1990 Census data — California housing has changed dramatically.
- Target capped at $500,001.
- District-level aggregation masks within-district variation.
- No individual property features (lot size, condition, etc.).

## 27 · How to Improve This Project

1. Add ocean_proximity as a feature (available in some versions).
2. Use spatial features (distance to coast, city center).
3. Try GBM with geospatial splits.
4. Log-transform the target to handle right skew.

## 28 · Production Considerations

- Need current ACS data, not 1990 Census.
- Property-level data (Zillow, Redfin APIs) would be more useful.
- Model should capture market dynamics (interest rates, supply).
- Confidence intervals needed for lending decisions.

## 29 · Common Mistakes

1. Not filling missing total_bedrooms before splitting.
2. Using raw total_rooms instead of per-household.
3. Ignoring geographic features or treating lat/lon as ordinal.
4. Not acknowledging 1990 data limitations.

## 30 · Mini Challenge / Exercises

1. Plot prediction errors on a geographic map.
2. Try clustering lat/lon into zones and use as categorical.
3. Log-transform target and compare R².
4. Add polynomial features for median_income.

## 31 · Final Summary / Key Takeaways

- Income and location dominate California housing prices.
- Per-household feature engineering significantly improves models.
- Boosting models achieve strong R² on this well-structured dataset.
- Geographic context matters — tabular models benefit from spatial features.

In [ ]:
metrics = {}
for name in top3_names:
    metrics[name] = all_results[name]
metrics['Baseline_LR'] = all_results['Baseline_LR']

with open(os.path.join(SAVE_DIR, 'metrics.json'), 'w') as f:
    json.dump(metrics, f, indent=2)
print('Metrics saved to metrics.json')
print('\nNotebook complete.')